# Rice Type Prediction

I will write a model that can predict types of rice. I am going to use a CNN model with attention layers. But first, I need to download the data and then I will prepare it to train the model. You can find the data in Kaggle.

In [ ]:
#I am importing libraries
import cv2
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

In [ ]:
#My dataset has 5 classes
labels=["Arborio", "Basmati", "Ipsala", "Jasmine", "Karacadag"]
img_path="/kaggle/input/rice-images-dataset/"

In [ ]:
#This is a for loop that creates a list of images and labels
img_list=[]
label_list=[]
for label in labels:
    for img_file in os.listdir(img_path+label):
        img_list.append(img_path+label+'/'+img_file)
        label_list.append(label)

In [ ]:
#I am transforming the list to dataframe
df=pd.DataFrame({"image":img_list,"label":label_list})
df.head()

,image,label
0,/kaggle/input/rice-images-dataset/Arborio/Arbo...,Arborio
1,/kaggle/input/rice-images-dataset/Arborio/Arbo...,Arborio
2,/kaggle/input/rice-images-dataset/Arborio/Arbo...,Arborio
3,/kaggle/input/rice-images-dataset/Arborio/Arbo...,Arborio
4,/kaggle/input/rice-images-dataset/Arborio/Arbo...,Arborio


In [ ]:
# If you have enough memory, you can make this cell comment before running it
df=df.sample(5000)

In [ ]:
#I am encoding the labels
d={"Arborio":0,"Basmati":1,"Ipsala":2,"Jasmine":3,"Karacadag":4}

In [ ]:
df["encoded_label"]=df["label"].map(d)

In [ ]:
##!!!This step is very important!!!##
x=[]
for img in df["image"]:
    img=cv2.imread(str(img)) #That read the image
    img=cv2.resize(img,(170,170)) #That resize the image to 170x170
    img=img/255.0 #That normalize the image-->Normalizing makes the rgb values between 0 and 1. That facilitates our computers to process
    x.append(img)

In [ ]:
#I have to convert the list to array because keras model takes array
x=np.array(x)

In [ ]:
#My target is encoded_label
y=df["encoded_label"]

In [ ]:
#I am splitting my dataset-->x_train,x_test,y_train,y_test
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, Dense, Flatten, Input, MaxPooling2D, Dropout, BatchNormalization, Add, Activation, GlobalAveragePooling2D, Reshape, Multiply
from tensorflow.keras import Model

In [ ]:
def cbam_block(input_tensor, reduction=16):
    channels = input_tensor.shape[-1]

    x = GlobalAveragePooling2D()(input_tensor)
    x = Dense(channels // reduction, activation='relu')(x)
    x = Dense(channels, activation='sigmoid')(x)
    x = Reshape((1, 1, channels))(x)
    x = Multiply()([input_tensor, x])


    y = Conv2D(1, (7, 7), padding='same', activation='sigmoid')(x)
    y = Multiply()([x, y])

    return y

inputs = Input(shape=(170, 170, 3))

x = Conv2D(32, (3,3), activation='relu')(inputs)
x = cbam_block(x)
x = MaxPooling2D(pool_size=(2,2))(x)

x = Conv2D(64, (3,3), activation='relu')(x)
x = cbam_block(x)
x = MaxPooling2D(pool_size=(2,2))(x)

x = Conv2D(80, (3,3), activation='relu')(x)
x = cbam_block(x)
x = MaxPooling2D(pool_size=(2,2))(x)

x = Conv2D(100, (3,3), activation='relu')(x)
x = cbam_block(x)

x = Flatten()(x)
x = Dense(128)(x)
outputs = Dense(5, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [ ]:
history=model.fit(x_train,y_train,epochs=10,validation_data=(x_test,y_test))

Epoch 1/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 29s 98ms/step - accuracy: 0.5725 - loss: 1.0063 - val_accuracy: 0.9370 - val_loss: 0.1761
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 0.9397 - loss: 0.1783 - val_accuracy: 0.9500 - val_loss: 0.1551
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - accuracy: 0.9506 - loss: 0.1328 - val_accuracy: 0.9570 - val_loss: 0.1306
Epoch 4/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - accuracy: 0.9609 - loss: 0.1109 - val_accuracy: 0.9570 - val_loss: 0.1097
Epoch 5/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - accuracy: 0.9653 - loss: 0.0933 - val_accuracy: 0.9570 - val_loss: 0.1280
Epoch 6/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - accuracy: 0.9647 - loss: 0.0946 - val_accuracy: 0.9700 - val_loss: 0.1089
Epoch 7/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - accuracy: 0.9739 - loss: 0.0712 - val_accuracy: 0.9620 - val_loss: 0.1106
Epoch 8/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - accuracy: 0.9647 - loss: 0.0926 - val_acc

In [ ]:
#I am saving my model for making my streamlit app
model.save("my_rice_prediction_model.h5")